In [2]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 00:03:05--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:ee00:b:20a5:b140:21, 2600:9000:234c:e800:b:20a5:b140:21, 2600:9000:234c:4600:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:ee00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  20.6MB/s    in 3.3s    

2026-03-08 00:03:08 (20.4 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [2]:
import os

os.listdir(".")

['yellow_tripdata_2025-11.parquet',
 'Untitled.ipynb',
 'yellow_2025_11_repart',
 'zoomcamp_env',
 'Homework6.ipynb',
 'README.md',
 'taxi_zone_lookup.csv',
 'taxi_zone_lookup.csv.1',
 '.ipynb_checkpoints']

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("YellowNovember") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 18:38:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [5]:
"""
### Question 2: Yellow November 2025
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB
"""
df_repart = df.repartition(4)

df_repart.write.mode("overwrite").parquet("yellow_2025_11_repart")

In [6]:
ls -lh yellow_2025_11_repart/

total 205312
-rw-r--r--  1 niteshmishra  staff     0B Mar  8 00:09 _SUCCESS
-rw-r--r--  1 niteshmishra  staff    24M Mar  8 00:09 part-00000-63162b34-b118-40ad-868e-55727140ced9-c000.snappy.parquet
-rw-r--r--  1 niteshmishra  staff    24M Mar  8 00:09 part-00001-63162b34-b118-40ad-868e-55727140ced9-c000.snappy.parquet
-rw-r--r--  1 niteshmishra  staff    24M Mar  8 00:09 part-00002-63162b34-b118-40ad-868e-55727140ced9-c000.snappy.parquet
-rw-r--r--  1 niteshmishra  staff    24M Mar  8 00:09 part-00003-63162b34-b118-40ad-868e-55727140ced9-c000.snappy.parquet


In [18]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [11]:
"""
Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604
- 225,768
"""
from pyspark.sql.functions import col

df_15 = df.filter(
    (col("tpep_pickup_datetime") >= "2025-11-15 00:00:00") &
    (col("tpep_pickup_datetime") <  "2025-11-16 00:00:00")  
)
count_15 = df_15.count()
print(count_15)

162604


In [21]:
"""
Question 4: Longest trip
What is the length of the longest trip in the dataset in hours?

22.7
58.2
90.6
134.5
"""
from pyspark.sql import functions as F

df_with_trip_duration = df.withColumn("trip_duration_hours", 
                                      (F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime")))/3600
                                     )
df_agg = df_with_trip_duration.agg(F.max("trip_duration_hours"))
df_agg.show()

+------------------------+
|max(trip_duration_hours)|
+------------------------+
|       90.64666666666666|
+------------------------+



In [5]:
"""
Question 6: Least frequent pickup location zone
Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

Governor's Island/Ellis Island/Liberty Island
Arden Heights
Rikers Island
Jamaica Bay
If multiple answers are correct, select any
"""
from pyspark.sql import functions as F

# Step 1 — Download the zone lookup file
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

# Step 2 — Read it into Spark
zones = (
    spark.read
        .option("header", "true")
        .csv("taxi_zone_lookup.csv")
)

# Step 3 — Join with taxi dataset
df_joined = df.join(
                zones,
                df.PULocationID == zones.LocationID,
                "left"
            )
# Step 4 — Count trips per pickup zone
# This sorts zones from least frequent → most frequent.
zone_counts = (
    df_joined
    .groupBy("Zone")
    .agg(F.count("*").alias("trip_count"))
    .orderBy("trip_count")
)

zone_counts.show(10, truncate=False)


--2026-03-08 18:40:53--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:3800:b:20a5:b140:21, 2600:9000:234c:e600:b:20a5:b140:21, 2600:9000:234c:5400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:3800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.3’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-08 18:40:53 (11.4 MB/s) - ‘taxi_zone_lookup.csv.3’ saved [12331/12331]

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Eltingville/Annadale/Prince's Bay            |1         |
|Arden Heights         